# QICK Pyro server (runs ON the ZCU216, in the board's Jupyter)

Run the cells top to bottom, **once per board boot**. The last cell blocks forever; that is expected, because it is the server.
After that, all experiment code runs on your PC with `rfsoc_client.py` and this notebook is not touched again.

Notes
- `start_server()` creates the `QickSoc` itself. **Do not** also create `QickSoc(...)` in this kernel.
- To stop the server: Kernel -> Interrupt, then Kernel -> **Restart** before starting it again (the firmware must be reloaded cleanly).
- The nameserver runs in a separate process (started in cell 2) so it survives a server restart.


In [1]:
# ensuring pyro files are seen, besides python from pynq
import sys
!{sys.executable} -m pip install --no-index --find-links=/home/xilinx/pyro_wheels Pyro4 serpent

Looking in links: /home/xilinx/pyro_wheels


In [2]:
# Cell 1 (optional, one-time): repair the firmware symlinks, as in your original notebook.
# Leave FIX_SYMLINKS = False once the links exist.
FIX_SYMLINKS = False

import os
from pathlib import Path

BASE = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_rfbv2_16mixmux16")
OUT_DIR = BASE / "out"
BITFILE = str(OUT_DIR / "qick_216_rfbv2.bit")

if FIX_SYMLINKS:
    for name in ["qick_216_rfbv2.bit", "qick_216_rfbv2.hwh"]:
        f = OUT_DIR / name
        if f.exists() or f.is_symlink():
            f.unlink()
    os.symlink("../top/top.runs/impl_1/d_1_wrapper.bit", str(OUT_DIR / "qick_216_rfbv2.bit"))
    os.symlink("../top/top.gen/sources_1/bd/d_1/hw_handoff/d_1.hwh", str(OUT_DIR / "qick_216_rfbv2.hwh"))
    print("Symlinks recreated.")
print("Bitfile:", BITFILE, "| exists:", os.path.exists(BITFILE))


Bitfile: /home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_216_rfbv2_16mixmux16/out/qick_216_rfbv2.bit | exists: True


In [3]:
# Cell 2: start the Pyro4 nameserver in its own process (skipped if one is already running).
import subprocess, sys, time
import Pyro4

NS_PORT = 8888
PROXY_NAME = "myqick"

def ns_running():
    Pyro4.config.SERIALIZER = "pickle"
    try:
        Pyro4.locateNS(host="localhost", port=NS_PORT)
        return True
    except Exception:
        return False

if not ns_running():
    ns_proc = subprocess.Popen([
        sys.executable, "-c",
        f"from qick.pyro import start_nameserver; start_nameserver(ns_host='0.0.0.0', ns_port={NS_PORT})"
    ])
    time.sleep(3)
print("Nameserver reachable:", ns_running())


Nameserver reachable: False


In [ ]:
# Cell 3: load the firmware and serve it. THIS CELL BLOCKS (the server loop).
# Extra keyword arguments go straight to QickSoc(...); these match your original notebook.
from qick.pyro import start_server

start_server(ns_host="localhost", ns_port=NS_PORT, proxy_name=PROXY_NAME,
             bitfile=BITFILE, external_clk=True)


looking for nameserver . . .
found nameserver
configuring reference clock chips, as requested
LMK04828 clock reference = 245.760 MHz, LMX2594 clock synth = 491.520 MHz
initialized QICK
registered QICK
registered member <qick.streamer.DataStreamer object at 0xffff7c1b49d0>
registered member ---------------------------------------------
 TPROC V2 INFO 
---------------------------------------------
Configuration:
fifo_depth    : 512
call_depth    : 255
pmem_size     : 4096
dmem_size     : 16384
wmem_size     : 1024
dreg_qty      : 16
in_port_qty   : 8
out_trig_qty  : 19
out_dport_qty : 1
out_dport_dw  : 8
out_wport_qty : 16
----------
Control:
has_io_ctrl   : YES
has_ext_flag  : NO
----------
Peripherals:
has_lfsr      : YES
has_divider   : YES
has_arith     : YES
has_time_read : YES
has_qnet      : NO
has_qcom      : NO
has_custom_periph: Only PA
----------
Debug:
debug         : NO
----------
Clocks:
CORE CLK : 215.04Mhz
TIME CLK : 430.08Mhz
----------

starting daemon
